# NeuroGraph · Phase 2 · Preprocess all subjects (step 2.4)
Settings: Accelerator **None**, Internet **On**. Run with **Save Version → Save & Run All (Commit)**.

## 1. Protect preinstalled packages

In [ ]:
from importlib.metadata import version

def write_constraints(pkgs: list[str], path: str) -> None:
    with open(path, "w") as f:
        f.write("\n".join(f"{p}=={version(p)}" for p in pkgs))

write_constraints(["torch", "numpy", "scipy", "scikit-learn", "mne"], "/tmp/keep.txt")
print(open("/tmp/keep.txt").read())

## 2. Get the latest code + install

In [ ]:
import os
R = "/kaggle/working/neurograph"
if not os.path.isdir(R):
    !git clone -q https://github.com/stuBBornButterfly/neurograph.git {R}
!git -C {R} pull -q
!git -C {R} log --oneline -1
%cd {R}
!pip install -q -r requirements.txt -c /tmp/keep.txt
!pip install -q -e . -c /tmp/keep.txt

## 3. Download all subjects to /tmp

In [ ]:
!python -m scripts.download_data --include "sub-*" 2>&1 | tail -4

## 4. Rebuild labels.csv

In [ ]:
!python -m scripts.summarize_dataset 2>&1 | tail -3

## 5. sub-001 check (prints most repaired channels)

In [ ]:
!python -m scripts.process_subject --subject 001

## 6. Batch: all 120 subjects (~2 h, resumable)

In [ ]:
!python -m scripts.run_preprocess

## 7. QC summary

In [ ]:
!python -m scripts.qc_summary

## 8. QC figure + output size

In [ ]:
from IPython.display import Image, display
display(Image("/kaggle/working/neurograph/reports/figures/qc_summary.png"))
!du -sh /kaggle/working/derivatives/*